# 02. Data Preprocessing Pipeline
**ML-Powered Intrusion Detection System (IDS) for Secure Network Monitoring**

This notebook demonstrates the modular, leakage-free data preprocessing pipeline for the CICIoT2023 dataset.

## 1. Imports & Path Configuration

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in sys.path
ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.preprocessing.load_data import discover_raw_files, load_raw_dataset
from src.preprocessing.clean_data import clean_dataset
from src.preprocessing.label_processing import identify_label_column, encode_labels
from src.preprocessing.split_data import split_dataset
from src.preprocessing.feature_processing import FeatureProcessor
from src.preprocessing.preprocess_pipeline import run_pipeline

RAW_DIR = ROOT_DIR / "data" / "raw"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"

## 2. Load Dataset & Inspect Raw Dimensions

In [ ]:
raw_files = discover_raw_files(RAW_DIR)
print(f"Discovered raw files: {[f.name for f in raw_files]}")

if raw_files:
    raw_df = load_raw_dataset(RAW_DIR)
    print(f"Raw dataset shape: {raw_df.shape}")
    display(raw_df.head())
else:
    print("No raw dataset files found in data/raw/. Please place CICIoT2023 CSV files in data/raw/.")

## 3. Data Cleaning (Infinities, Nulls, Duplicates, Constants)

In [ ]:
if raw_files:
    cleaned_df, clean_stats = clean_dataset(raw_df)
    print("Cleaning Summary:", clean_stats)
    print(f"Cleaned dataset shape: {cleaned_df.shape}")

## 4. Label Identification & Encoding

In [ ]:
if raw_files:
    label_col = identify_label_column(cleaned_df)
    mapping_path = PROCESSED_DIR / "label_mapping.json"
    y_encoded, label_mapping = encode_labels(cleaned_df, label_column=label_col, output_mapping_path=mapping_path)
    X_unscaled = cleaned_df.drop(columns=[label_col])
    print(f"Identified target column: '{label_col}'")
    print("Label Mapping:", label_mapping)

## 5. Stratified Train / Validation / Test Splitting

In [ ]:
if raw_files:
    X_train, X_val, X_test, y_train, y_val, y_test, split_meta = split_dataset(
        X_unscaled, y_encoded, train_size=0.70, val_size=0.15, test_size=0.15, random_state=42, stratify=True
    )
    print("Split Metadata:", split_meta)
    print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")

## 6. Leakage-Free Feature Scaling (Fit on Train ONLY)

In [ ]:
if raw_files:
    processor = FeatureProcessor(scaler_type="StandardScaler", artifact_dir=ROOT_DIR / "models" / "preprocessing")
    X_train_scaled, feat_meta = processor.fit_transform(X_train, save_artifacts=True)
    X_val_scaled = processor.transform(X_val)
    X_test_scaled = processor.transform(X_test)
    print(f"Scaled Feature Dimension: {X_train_scaled.shape[1]}")
    print("Feature Processor Metadata:", feat_meta)

## 7. Class Distribution Analysis

In [ ]:
if raw_files:
    plt.figure(figsize=(10, 4))
    y_train.value_counts().plot(kind='bar', color='skyblue')
    plt.title('Training Class Distribution')
    plt.xlabel('Encoded Class')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

## 8. Final Processed Feature Matrix & Verification

In [ ]:
if raw_files:
    assert not np.isnan(X_train_scaled).any(), "NaN detected in X_train"
    assert not np.isnan(X_val_scaled).any(), "NaN detected in X_val"
    assert not np.isnan(X_test_scaled).any(), "NaN detected in X_test"
    assert X_train_scaled.shape[1] == X_val_scaled.shape[1] == X_test_scaled.shape[1], "Shape mismatch"
    print("[SUCCESS] All automated preprocessing verifications passed!")